# Stack & Queue — Démos pratiques

> La théorie est dans [01_theorie_stack_queue.md](01_theorie_stack_queue.md).

Tout est fait avec du code simple : des `list`, des `deque` et quelques petites fonctions. **Pas de classe** : on se concentre sur le concept.

**Au programme**

| Partie | Contenu |
|---|---|
| 🥞 Pile | démo 1 : pile avec une `list` · démo 2 : bouton « Précédent » du navigateur · démo 3 : annuler / rétablir |
| 🚶 File | démo 4 : file d'attente avec `deque` · démo 5 : `list.pop(0)` contre `deque.popleft()` · démo 6 : file d'impression |
| 🧰 Variantes | démo 7 : `deque(maxlen=...)` · démo 8 : file de priorité avec `heapq` |
| ✍️ Exercices | 6 exercices de groupe façon LeetCode + solutions à la fin |

👉 Exécutez les cellules **dans l'ordre**.

---
# 🥞 Partie 1 — La pile (Stack)

## 🟢 Démo 1 — Une pile avec une simple `list`

Rappel : une pile, c'est une **pile d'assiettes**. On pose sur le dessus, on reprend sur le dessus (**LIFO**).
Avec une `list`, le **sommet** de la pile est la **fin** de la liste.

In [ ]:
# Étape 1 : on crée une pile vide (une simple liste)
plate_stack = []

# Étape 2 : on empile trois assiettes avec append() → elles vont à la FIN de la liste (= le sommet)
plate_stack.append("assiette bleue")
plate_stack.append("assiette verte")
plate_stack.append("assiette rouge")
print("Pile après 3 ajouts :", plate_stack)

# Étape 3 : on regarde le sommet SANS le retirer → l'élément d'indice -1 (le dernier)
print("Sommet de la pile :", plate_stack[-1])

# Étape 4 : on dépile avec pop() → retire ET renvoie le dernier élément ajouté
removed_plate = plate_stack.pop()
print("Assiette retirée :", removed_plate)          # la rouge : dernière entrée, première sortie
print("Pile après un retrait :", plate_stack)

# Étape 5 : on vide la pile tant qu'elle n'est pas vide
# "while plate_stack" signifie "tant que la liste n'est pas vide" (une liste vide vaut False)
while plate_stack:
    print("On retire :", plate_stack.pop())

print("Pile vide ?", len(plate_stack) == 0)

In [ ]:
# ⚠️ Piège : dépiler une pile VIDE provoque une erreur IndexError
empty_stack = []

try:
    empty_stack.pop()
except IndexError as error:
    # On attrape l'erreur pour que le notebook continue, et on affiche le message de Python
    print("Erreur :", error)

## 🟢 Démo 2 — Le bouton « Précédent » du navigateur

Chaque page visitée est **empilée**. Cliquer sur « Précédent » **dépile** la page actuelle : on revient sur celle d'en dessous.

In [ ]:
# Étape 1 : la pile de l'historique, vide au départ
page_history = []

# Étape 2 : l'utilisateur visite trois pages → on les empile une par une
page_history.append("accueil")
page_history.append("produits")
page_history.append("panier")
print("Historique (bas → sommet) :", page_history)

# Étape 3 : la page actuelle est le sommet de la pile
print("Page actuelle :", page_history[-1])

# Étape 4 : clic sur "Précédent" → on dépile la page actuelle
left_page = page_history.pop()
print("On quitte :", left_page, "→ on revient sur :", page_history[-1])

# Étape 5 : encore "Précédent"
left_page = page_history.pop()
print("On quitte :", left_page, "→ on revient sur :", page_history[-1])

# Étape 6 : on est sur la première page → on vérifie qu'il reste une page AVANT de dépiler
if len(page_history) > 1:
    page_history.pop()
else:
    print("Impossible de revenir en arrière : on est sur la première page (", page_history[-1], ")")

## 🔵 Démo 3 — Exemple concret : annuler / rétablir (Ctrl+Z / Ctrl+Y)

Un éditeur de texte utilise **deux piles** (deux simples listes) :
- `undo_stack` : les versions **précédentes** du texte (pour annuler) ;
- `redo_stack` : les versions **annulées** (pour rétablir).

Règle : quand on écrit quelque chose de nouveau, on **vide** la pile « rétablir ».

In [ ]:
text = ""
undo_stack = []   # anciennes versions du texte, la plus récente au sommet
redo_stack = []   # versions annulées, pour pouvoir les rétablir

# Étape 1 : on écrit trois morceaux. Avant chaque modification, on empile l'ancienne version
for new_words in ["Bonjour", " tout", " le monde"]:
    undo_stack.append(text)
    text = text + new_words
    redo_stack.clear()   # nouvelle action → plus rien à rétablir
print("Texte :", text)
print("Pile annuler :", undo_stack)

# Étape 2 : Ctrl+Z → le texte actuel part dans redo_stack, on reprend le sommet de undo_stack
redo_stack.append(text)
text = undo_stack.pop()
print("Après 1 annulation :", text)

# Étape 3 : encore Ctrl+Z
redo_stack.append(text)
text = undo_stack.pop()
print("Après 2 annulations :", text)

# Étape 4 : Ctrl+Y → l'inverse : le texte actuel repart dans undo_stack, on reprend le sommet de redo_stack
undo_stack.append(text)
text = redo_stack.pop()
print("Après 1 rétablissement :", text)

# Étape 5 : on écrit quelque chose de nouveau → la pile rétablir est vidée
undo_stack.append(text)
text = text + " !"
redo_stack.clear()
print("Après une nouvelle écriture :", text)
print("Encore quelque chose à rétablir ?", len(redo_stack) > 0)

---
# 🚶 Partie 2 — La file (Queue)

## 🟢 Démo 4 — Une file d'attente avec `deque`

Rappel : une file, c'est la **file d'attente à la caisse** (**FIFO** : premier arrivé, premier servi).
On ajoute à la **fin** avec `append()`, on retire la **tête** avec `popleft()`.

In [ ]:
# deque vient du module collections (bibliothèque standard, rien à installer)
from collections import deque

# Étape 1 : une file vide
checkout_queue = deque()

# Étape 2 : trois clients arrivent, dans cet ordre → ils se placent à la FIN de la file
checkout_queue.append("Alice")
checkout_queue.append("Bob")
checkout_queue.append("Chloé")
print("File (tête → fin) :", list(checkout_queue))

# Étape 3 : qui est le prochain ? → la tête de file, indice 0 (on lit sans retirer)
print("Prochain client :", checkout_queue[0])

# Étape 4 : on sert les clients un par un → popleft() retire la TÊTE
while checkout_queue:
    served_customer = checkout_queue.popleft()
    print("On sert :", served_customer, "| il reste", len(checkout_queue), "client(s)")

## ⚔️ Démo 5 — `list.pop(0)` contre `deque.popleft()`

On pourrait faire une file avec une `list` et `pop(0)`, mais Python doit alors **décaler tous les éléments** (O(n)).
`deque.popleft()` retire la tête en **O(1)**. Chronométrons !

In [ ]:
# deque vient du module collections (bibliothèque standard, rien à installer)
from collections import deque
# perf_counter() renvoie un temps en secondes, très précis : idéal pour chronométrer
from time import perf_counter

number_of_customers = 50_000   # le "_" sert juste à lire le nombre plus facilement (50000)

# --- Version 1 : file avec une list et pop(0) ---
# Étape 1 : on remplit la liste avec 50 000 clients numérotés
customer_list = list(range(number_of_customers))
# Étape 2 : on chronomètre le service de tous les clients, du premier au dernier
start_time = perf_counter()
while customer_list:
    customer_list.pop(0)          # retire la tête → décale tous les autres éléments
list_duration = perf_counter() - start_time

# --- Version 2 : file avec un deque et popleft() ---
# Étape 3 : même remplissage, mais dans un deque
customer_deque = deque(range(number_of_customers))
# Étape 4 : même chronométrage
start_time = perf_counter()
while customer_deque:
    customer_deque.popleft()      # retire la tête sans rien décaler
deque_duration = perf_counter() - start_time

# Étape 5 : on compare (round() arrondit à 4 décimales pour un affichage lisible)
print("list.pop(0)      :", round(list_duration, 4), "secondes")
print("deque.popleft()  :", round(deque_duration, 4), "secondes")
print("deque est environ", round(list_duration / deque_duration), "fois plus rapide")

> 💡 Essayez avec `number_of_customers = 200_000` : le temps de la `list` explose (il grandit comme n²), celui du `deque` reste proportionnel à n.

## 🔵 Démo 6 — Exemple concret : une file d'impression

Plusieurs personnes envoient des documents à **une seule** imprimante. Elle imprime **dans l'ordre d'envoi**.
Chaque document est un dictionnaire `{"owner": ..., "document_name": ..., "page_count": ...}`.

In [ ]:
from collections import deque

pages_per_minute = 20
print_queue = deque()

# Étape 1 : trois personnes envoient un document, dans cet ordre → ajout à la fin de la file
print_queue.append({"owner": "Alice", "document_name": "facture.pdf", "page_count": 2})
print_queue.append({"owner": "Bob", "document_name": "rapport_annuel.pdf", "page_count": 60})
print_queue.append({"owner": "Chloé", "document_name": "billet_train.pdf", "page_count": 1})
print("Documents en attente :", len(print_queue))

# Étape 2 : l'imprimante traite la file, toujours le document en tête
elapsed_minutes = 0
while print_queue:
    print_job = print_queue.popleft()
    # Durée d'impression de ce document, en minutes
    elapsed_minutes = elapsed_minutes + print_job["page_count"] / pages_per_minute
    print("Impression de", print_job["document_name"], "pour", print_job["owner"],
          "- terminé à", round(elapsed_minutes, 2), "min")

# Étape 3 : Chloé n'a qu'une page, mais elle a attendu derrière le gros rapport de Bob → c'est le FIFO

---
# 🧰 Partie 3 — Les variantes

## 🟢 Démo 7 — `deque(maxlen=...)` : garder les N dernières actions

Avec `maxlen`, le deque a une **taille maximum** : quand il est plein, ajouter à droite **éjecte automatiquement** l'élément le plus ancien à gauche.
Pratique pour « les 3 derniers fichiers ouverts » ou « les 5 dernières erreurs ».

In [ ]:
from collections import deque

# Étape 1 : on veut se souvenir des 3 derniers fichiers ouverts seulement
recent_files = deque(maxlen=3)

# Étape 2 : on ouvre 5 fichiers, l'un après l'autre
for file_name in ["devis.odt", "contrat.pdf", "photo.png", "notes.txt", "budget.xlsx"]:
    recent_files.append(file_name)
    print("Ouverture de", file_name, "→ récents :", list(recent_files))

# Étape 3 : seuls les 3 derniers restent, les plus anciens ont été éjectés
print("Fichiers récents finaux :", list(recent_files))

## 🔵 Démo 8 — File de priorité avec `heapq` : tickets de support

Au support client, on ne traite pas « premier arrivé, premier servi » mais **le plus urgent d'abord**.
`heapq` garde toujours le **plus petit** élément en tête. On stocke des tuples `(priorite, numero_arrivee, description)` :
- `priorite` : 1 = critique, 2 = important, 3 = normal (plus c'est petit, plus c'est urgent) ;
- `numero_arrivee` : départage deux tickets de même priorité (le premier arrivé passe d'abord).

In [ ]:
import heapq

# Étape 1 : le "tas" est une simple liste vide, que heapq va organiser
support_tickets = []
arrival_number = 0   # compteur pour départager les priorités égales

# Étape 2 : des tickets arrivent dans le désordre
incoming_tickets = [
    (3, "Changer la couleur du logo"),
    (1, "Le site de vente est en panne"),
    (2, "Erreur dans une facture client"),
    (1, "Impossible de se connecter à Odoo"),
    (3, "Ajouter un champ dans un formulaire"),
]

for priority, description in incoming_tickets:
    arrival_number = arrival_number + 1
    # heappush ajoute l'élément et réorganise le tas — O(log n)
    heapq.heappush(support_tickets, (priority, arrival_number, description))
    print("Nouveau ticket (priorité", priority, ") :", description)

# Étape 3 : on traite les tickets → toujours le plus urgent en premier
print()
while support_tickets:
    # heappop retire et renvoie le PLUS PETIT tuple → la priorité la plus urgente
    priority, arrival_number, description = heapq.heappop(support_tickets)
    print("Traitement priorité", priority, "(arrivé n°", arrival_number, ") :", description)

---
# ✍️ Partie 4 — Exercices de groupe façon LeetCode

**Méthode** (détaillée dans la théorie) : 1. comprendre → 2. exemples à la main → 3. solution naïve → 4. optimiser → 5. coder → 6. tester.
**Rôles** : pilote, navigateur, testeur, rapporteur — on tourne à chaque exercice.

Chaque exercice contient : un énoncé, des exemples, un **squelette** à compléter et des **tests**.
Les tests affichent ✅ ou ❌ (ils ne plantent pas le notebook). Les **solutions commentées** sont à la fin.

| # | Exercice | Niveau | Structure | Inspiré de |
|---|---|---|---|---|
| 1 | Inverser une chaîne | Easy | pile | — |
| 2 | Parenthèses valides | Easy | pile | LeetCode 20 |
| 3 | Supprimer les doublons adjacents | Easy | pile | LeetCode 1047 |
| 4 | Compter les appels récents | Easy | file | LeetCode 933 |
| 5 | Une file avec deux piles | Easy | 2 piles | LeetCode 232 |
| 6 | Températures quotidiennes | Medium | pile | LeetCode 739 |

In [ ]:
def check(test_name, obtained_result, expected_result):
    """Petit outil de test : compare le résultat obtenu au résultat attendu et affiche ✅ ou ❌."""
    if obtained_result == expected_result:
        print("✅", test_name)
    else:
        print("❌", test_name, "→ obtenu :", obtained_result, "| attendu :", expected_result)

## Exercice 1 — Inverser une chaîne (Easy · pile)

Écrire `reverse_text(text)` qui renvoie le texte **à l'envers**, en utilisant une **pile** (pas de `text[::-1]` !).

| Entrée | Sortie |
|---|---|
| `"odoo"` | `"oodo"` |
| `"Python"` | `"nohtyP"` |
| `""` | `""` |

💡 **Indice** : on empile chaque lettre, puis on dépile tout. Dans quel ordre sortent les lettres ?

In [ ]:
def reverse_text(text):
    # À compléter
    pass

check("odoo", reverse_text("odoo"), "oodo")
check("Python", reverse_text("Python"), "nohtyP")
check("chaîne vide", reverse_text(""), "")
check("une lettre", reverse_text("a"), "a")

## Exercice 2 — Parenthèses valides (Easy · pile · LeetCode 20)

Écrire `is_valid_brackets(text)` qui renvoie `True` si toutes les parenthèses `()`, crochets `[]` et accolades `{}` sont **bien fermées, dans le bon ordre**.

| Entrée | Sortie | Pourquoi |
|---|---|---|
| `"()"` | `True` | |
| `"()[]{}"` | `True` | |
| `"([{}])"` | `True` | imbriquées correctement |
| `"(]"` | `False` | mauvais type de fermeture |
| `"([)]"` | `False` | fermées dans le mauvais ordre |
| `"(("` | `False` | jamais fermées |
| `""` | `True` | rien à fermer |

💡 **Indice** : *« je dois me souvenir de la dernière parenthèse ouverte »* → pile. Une fermante doit correspondre au **sommet**.

In [ ]:
def is_valid_brackets(text):
    # À compléter
    pass

check("()", is_valid_brackets("()"), True)
check("()[]{}", is_valid_brackets("()[]{}"), True)
check("([{}])", is_valid_brackets("([{}])"), True)
check("(]", is_valid_brackets("(]"), False)
check("([)]", is_valid_brackets("([)]"), False)
check("((", is_valid_brackets("(("), False)
check("))", is_valid_brackets("))"), False)
check("chaîne vide", is_valid_brackets(""), True)

## Exercice 3 — Supprimer les doublons adjacents (Easy · pile · LeetCode 1047)

Écrire `remove_adjacent_duplicates(text)` : tant que deux lettres **identiques sont côte à côte**, on les supprime toutes les deux. On renvoie le texte final.

| Entrée | Sortie | Déroulé |
|---|---|---|
| `"abbaca"` | `"ca"` | `abbaca` → `aaca` (on retire `bb`) → `ca` (on retire `aa`) |
| `"azxxzy"` | `"ay"` | `azxxzy` → `azzy` → `ay` |
| `"abc"` | `"abc"` | aucun doublon |

💡 **Indice** : on parcourt les lettres. Si la lettre est **égale au sommet** de la pile, on dépile ; sinon on empile. À la fin, la pile contient le résultat.

In [ ]:
def remove_adjacent_duplicates(text):
    # À compléter
    pass

check("abbaca", remove_adjacent_duplicates("abbaca"), "ca")
check("azxxzy", remove_adjacent_duplicates("azxxzy"), "ay")
check("abc", remove_adjacent_duplicates("abc"), "abc")
check("aa", remove_adjacent_duplicates("aa"), "")
check("chaîne vide", remove_adjacent_duplicates(""), "")

## Exercice 4 — Compter les appels récents (Easy · file · LeetCode 933)

Un serveur reçoit des requêtes. On a la liste des **instants** (en millisecondes, dans l'ordre croissant) où elles arrivent.
Pour **chaque** requête, on veut savoir combien de requêtes ont eu lieu dans les **3000 ms** qui précèdent (elle comprise), c'est-à-dire dans l'intervalle `[instant - 3000, instant]`.

Écrire `count_recent_requests(request_times)` qui renvoie la liste de ces nombres.

| Requête à | Requêtes dans la fenêtre | Nombre |
|---|---|---|
| 1 | `[1]` | 1 |
| 100 | `[1, 100]` | 2 |
| 3001 | `[1, 100, 3001]` | 3 |
| 3002 | `[100, 3001, 3002]` (1 est trop vieux) | 3 |
| 7000 | `[7000]` | 1 |

Entrée `[1, 100, 3001, 3002, 7000]` → sortie `[1, 2, 3, 3, 1]`.

💡 **Indice** : les requêtes arrivent dans l'ordre → **file**. On ajoute chaque requête à la fin ; les plus vieilles sont en **tête** : on les retire tant qu'elles sont trop anciennes.

In [ ]:
from collections import deque

def count_recent_requests(request_times):
    # À compléter
    pass

check("exemple", count_recent_requests([1, 100, 3001, 3002, 7000]), [1, 2, 3, 3, 1])
check("une requête", count_recent_requests([5]), [1])
check("toutes proches", count_recent_requests([10, 20, 30]), [1, 2, 3])
check("liste vide", count_recent_requests([]), [])

## Exercice 5 — Une file avec deux piles (Easy · LeetCode 232)

On veut une **file (FIFO)** en n'utilisant **que deux piles** (des `list` avec seulement `append`, `pop` et `[-1]`) :
- `input_stack` reçoit les nouveaux éléments (`input_stack.append(element)`) ;
- `output_stack` sert à les ressortir.

Écrire `dequeue_with_two_stacks(input_stack, output_stack)` qui retire et renvoie l'élément **le plus ancien**.

```
input_stack.append(1), input_stack.append(2), input_stack.append(3)
dequeue_with_two_stacks(...) → 1      (premier entré, premier sorti)
```

💡 **Indice** : si `output_stack` est vide, on **verse** toute `input_stack` dans `output_stack` (on dépile l'une, on empile dans l'autre) : l'ordre s'inverse, et le plus ancien se retrouve au sommet. Ensuite, on dépile `output_stack`.

> ℹ️ Les listes sont **modifiées directement** par la fonction : une `list` passée en paramètre n'est pas copiée, la fonction travaille sur la même liste.

In [ ]:
def dequeue_with_two_stacks(input_stack, output_stack):
    # À compléter
    pass

input_stack = []
output_stack = []
input_stack.append(1)
input_stack.append(2)
input_stack.append(3)
check("dequeue → 1", dequeue_with_two_stacks(input_stack, output_stack), 1)
input_stack.append(4)
check("dequeue → 2", dequeue_with_two_stacks(input_stack, output_stack), 2)
check("dequeue → 3", dequeue_with_two_stacks(input_stack, output_stack), 3)
check("dequeue → 4", dequeue_with_two_stacks(input_stack, output_stack), 4)
check("tout est vide", input_stack == [] and output_stack == [], True)

## Exercice 6 — Températures quotidiennes (Medium · pile · LeetCode 739)

On reçoit la liste des températures de chaque jour. Pour chaque jour, combien de jours faut-il **attendre** pour avoir une température **plus chaude** ? S'il n'y en a jamais, on met `0`.

| Entrée | Sortie |
|---|---|
| `[73, 74, 75, 71, 69, 72, 76, 73]` | `[1, 1, 4, 2, 1, 1, 0, 0]` |
| `[30, 40, 50, 60]` | `[1, 1, 1, 0]` |
| `[30, 20, 10]` | `[0, 0, 0]` |

Lecture du 1er exemple : le jour 0 (73°) attend 1 jour pour 74° ; le jour 2 (75°) attend 4 jours pour 76°.

💡 **Indice 1 (naïf)** : pour chaque jour, regarder les jours suivants un par un → O(n²).
💡 **Indice 2 (optimisé, O(n))** : on garde dans une pile les **indices des jours qui attendent encore** un jour plus chaud. Pour chaque nouveau jour, tant qu'il est **plus chaud** que le jour au sommet de la pile, ce jour au sommet a trouvé sa réponse : on le dépile et on calcule l'écart.

In [ ]:
def days_until_warmer(temperatures):
    # À compléter
    pass

check("exemple 1", days_until_warmer([73, 74, 75, 71, 69, 72, 76, 73]), [1, 1, 4, 2, 1, 1, 0, 0])
check("croissant", days_until_warmer([30, 40, 50, 60]), [1, 1, 1, 0])
check("décroissant", days_until_warmer([30, 20, 10]), [0, 0, 0])
check("un seul jour", days_until_warmer([50]), [0])

---
# ✅ Solutions commentées

> ⚠️ À ouvrir **après** avoir cherché en groupe ! Chaque solution redéfinit la fonction puis relance les tests.

### Solution 1 — Inverser une chaîne · O(n)

In [ ]:
def reverse_text(text):
    letter_stack = []
    # Étape 1 : on empile chaque lettre, de la première à la dernière
    for letter in text:
        letter_stack.append(letter)
    # Étape 2 : on dépile tout → les lettres sortent dans l'ordre inverse (LIFO)
    reversed_text = ""
    while letter_stack:
        reversed_text = reversed_text + letter_stack.pop()
    return reversed_text

check("odoo", reverse_text("odoo"), "oodo")
check("Python", reverse_text("Python"), "nohtyP")
check("chaîne vide", reverse_text(""), "")
check("une lettre", reverse_text("a"), "a")

### Solution 2 — Parenthèses valides · O(n)

In [ ]:
def is_valid_brackets(text):
    # Dictionnaire : pour chaque symbole FERMANT, le symbole OUVRANT qui lui correspond
    matching_opening = {")": "(", "]": "[", "}": "{"}
    opening_stack = []

    for character in text:
        if character in "([{":
            # Étape 1 : une ouvrante → on la met de côté au sommet de la pile
            opening_stack.append(character)
        elif character in matching_opening:
            # Étape 2 : une fermante → la pile ne doit pas être vide...
            if not opening_stack:
                return False
            # ... et le sommet doit être l'ouvrante correspondante
            last_opening = opening_stack.pop()
            if last_opening != matching_opening[character]:
                return False

    # Étape 3 : à la fin, toutes les ouvrantes doivent avoir été fermées → pile vide
    return len(opening_stack) == 0

check("()", is_valid_brackets("()"), True)
check("()[]{}", is_valid_brackets("()[]{}"), True)
check("([{}])", is_valid_brackets("([{}])"), True)
check("(]", is_valid_brackets("(]"), False)
check("([)]", is_valid_brackets("([)]"), False)
check("((", is_valid_brackets("(("), False)
check("))", is_valid_brackets("))"), False)
check("chaîne vide", is_valid_brackets(""), True)

### Solution 3 — Supprimer les doublons adjacents · O(n)

In [ ]:
def remove_adjacent_duplicates(text):
    letter_stack = []
    for letter in text:
        # Étape 1 : si la lettre est identique au sommet, les deux s'annulent → on dépile
        if letter_stack and letter_stack[-1] == letter:
            letter_stack.pop()
        # Étape 2 : sinon on empile la lettre
        else:
            letter_stack.append(letter)
    # Étape 3 : on recolle les lettres restantes ("".join met les éléments bout à bout)
    return "".join(letter_stack)

check("abbaca", remove_adjacent_duplicates("abbaca"), "ca")
check("azxxzy", remove_adjacent_duplicates("azxxzy"), "ay")
check("abc", remove_adjacent_duplicates("abc"), "abc")
check("aa", remove_adjacent_duplicates("aa"), "")
check("chaîne vide", remove_adjacent_duplicates(""), "")

### Solution 4 — Compter les appels récents · O(n)

In [ ]:
from collections import deque

def count_recent_requests(request_times):
    recent_requests = deque()   # file des requêtes encore dans la fenêtre, la plus ancienne en tête
    counts = []                 # résultat : un nombre par requête

    for request_time in request_times:
        # Étape 1 : la nouvelle requête entre à la fin de la file
        recent_requests.append(request_time)
        # Étape 2 : on retire en tête les requêtes plus vieilles que 3000 ms
        oldest_allowed_time = request_time - 3000
        while recent_requests[0] < oldest_allowed_time:
            recent_requests.popleft()
        # Étape 3 : ce qui reste dans la file est dans la fenêtre
        counts.append(len(recent_requests))

    return counts

check("exemple", count_recent_requests([1, 100, 3001, 3002, 7000]), [1, 2, 3, 3, 1])
check("une requête", count_recent_requests([5]), [1])
check("toutes proches", count_recent_requests([10, 20, 30]), [1, 2, 3])
check("liste vide", count_recent_requests([]), [])

### Solution 5 — Une file avec deux piles · O(1) en moyenne par opération

In [ ]:
def dequeue_with_two_stacks(input_stack, output_stack):
    # Étape 1 : si output_stack est vide, on y verse toute input_stack
    # (on ne verse QUE dans ce cas, sinon on mélangerait l'ordre)
    if not output_stack:
        while input_stack:
            # On dépile input_stack et on empile dans output_stack → l'ordre s'inverse,
            # le premier élément entré se retrouve au sommet de output_stack
            output_stack.append(input_stack.pop())

    # Étape 2 : si les deux piles sont vides, la file est vide
    if not output_stack:
        print("La file est vide")
        return None

    # Étape 3 : le sommet de output_stack est l'élément le plus ancien
    return output_stack.pop()

input_stack = []
output_stack = []
input_stack.append(1)
input_stack.append(2)
input_stack.append(3)
check("dequeue → 1", dequeue_with_two_stacks(input_stack, output_stack), 1)
input_stack.append(4)
check("dequeue → 2", dequeue_with_two_stacks(input_stack, output_stack), 2)
check("dequeue → 3", dequeue_with_two_stacks(input_stack, output_stack), 3)
check("dequeue → 4", dequeue_with_two_stacks(input_stack, output_stack), 4)
check("tout est vide", input_stack == [] and output_stack == [], True)

> 💡 Pourquoi « O(1) en moyenne » ? Chaque élément est versé d'une pile à l'autre **une seule fois** dans sa vie. Un retrait peut parfois coûter cher (on verse tout), mais sur l'ensemble des opérations, le coût par opération reste constant : on parle de complexité **amortie**.

### Solution 6 — Températures quotidiennes

In [ ]:
# Version naïve O(n²) : pour chaque jour, on regarde tous les jours suivants
def days_until_warmer_naive(temperatures):
    waiting_days = [0] * len(temperatures)   # par défaut : 0 (jamais plus chaud)
    for current_day in range(len(temperatures)):
        for next_day in range(current_day + 1, len(temperatures)):
            if temperatures[next_day] > temperatures[current_day]:
                waiting_days[current_day] = next_day - current_day
                break   # on a trouvé le premier jour plus chaud, on arrête de chercher
    return waiting_days


# Version optimisée O(n) avec une pile d'indices de jours "en attente"
def days_until_warmer(temperatures):
    waiting_days = [0] * len(temperatures)
    # Pile des indices des jours qui n'ont pas encore trouvé de jour plus chaud
    waiting_day_stack = []

    for current_day in range(len(temperatures)):
        current_temperature = temperatures[current_day]
        # Étape 1 : tant que le jour actuel est plus chaud que le jour au sommet de la pile,
        # ce jour au sommet a enfin trouvé sa réponse
        while waiting_day_stack and current_temperature > temperatures[waiting_day_stack[-1]]:
            previous_day = waiting_day_stack.pop()
            waiting_days[previous_day] = current_day - previous_day
        # Étape 2 : le jour actuel attend à son tour un jour plus chaud
        waiting_day_stack.append(current_day)

    # Les jours restés dans la pile n'ont jamais trouvé plus chaud → ils gardent 0
    return waiting_days

for function_to_test in [days_until_warmer_naive, days_until_warmer]:
    print("Tests de", function_to_test.__name__)
    check("exemple 1", function_to_test([73, 74, 75, 71, 69, 72, 76, 73]), [1, 1, 4, 2, 1, 1, 0, 0])
    check("croissant", function_to_test([30, 40, 50, 60]), [1, 1, 1, 0])
    check("décroissant", function_to_test([30, 20, 10]), [0, 0, 0])
    check("un seul jour", function_to_test([50]), [0])

In [ ]:
# Bonus : on chronomètre les deux versions sur 5 000 jours de températures aléatoires
import random
from time import perf_counter

# Étape 1 : on génère 5 000 températures entre 0 et 40 degrés
random_temperatures = [random.randint(0, 40) for day in range(5_000)]

# Étape 2 : version naïve
start_time = perf_counter()
naive_result = days_until_warmer_naive(random_temperatures)
naive_duration = perf_counter() - start_time

# Étape 3 : version avec pile
start_time = perf_counter()
stack_result = days_until_warmer(random_temperatures)
stack_duration = perf_counter() - start_time

# Étape 4 : mêmes résultats, mais pas le même temps
print("Mêmes résultats ?", naive_result == stack_result)
print("Naïve O(n²) :", round(naive_duration, 4), "secondes")
print("Pile  O(n)  :", round(stack_duration, 4), "secondes")

---
# 🎯 Résumé

- 🥞 **Pile** (`list`) : `append` / `pop` / `[-1]` → annuler, historique, parenthèses, « dernier ouvert ».
- 🚶 **File** (`deque`) : `append` / `popleft` / `[0]` → ordre d'arrivée, impression, fenêtre de temps.
- 🧰 `deque(maxlen=n)` pour garder les n derniers, `heapq` pour traiter **le plus urgent** d'abord.
- En exercice, chercher le mot-clé : *« dernier vu / ouvert »* → pile, *« le plus ancien / ordre d'arrivée »* → file.